<a href="https://colab.research.google.com/github/Andci/AdWords_Bot/blob/main/FLUX_1_schnell_Runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# install the packages
!pip install git+https://github.com/huggingface/diffusers.git
!pip install transformers sentencepiece accelerate protobuf

In [ ]:
import torch
from diffusers import FluxPipeline
import diffusers
from PIL import Image
import matplotlib.pyplot as plt

# Modify the rope function to handle CUDA device
_flux_rope = diffusers.models.transformers.transformer_flux.rope
def new_flux_rope(pos: torch.Tensor, dim: int, theta: int) -> torch.Tensor:
    assert dim % 2 == 0, "The dimension must be even."
    if pos.device.type == "cuda":
        # Move tensor to CPU for ROPE computation, then move it back to CUDA
        return _flux_rope(pos.to("cpu"), dim, theta).to(device=pos.device)
    else:
        # Perform ROPE computation directly if tensor is not on CUDA
        return _flux_rope(pos, dim, theta)
diffusers.models.transformers.transformer_flux.rope = new_flux_rope

# Load the Flux Schnell model
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    revision='refs/pr/1',
    torch_dtype=torch.bfloat16
).to("cuda")

# Define the prompt
# This is the textual description that the model will use to generate the image
prompt = "Create an image of a middle-aged couple, a man and a woman in their 40s, drinking water from a water bottle in a lush natural setting. The man has short brown hair and is wearing a blue T-shirt and hiking shorts. The woman has her blonde hair tied back in a ponytail and is dressed in a green T-shirt and hiking pants. They are standing on a forest trail surrounded by dense greenery and wildflowers. Sunlight filters through the treetops, casting a warm, dappled light on the scene, which evokes a sense of vitality and connection with nature."

# Generate the image
out = pipe(
    prompt=prompt,
    guidance_scale=0.,
    height=1024,
    width=1024,
    num_inference_steps=4,
    max_sequence_length=256,
).images[0]

# Save the generated image
out.save("gen_image.png")

# Display the generated image
image = Image.open("gen_image.png")
plt.imshow(image)
plt.axis('off')  # Hide axes
plt.show()